In [1]:
from transformers import pipeline

# 모델 로드

### sentiment-analysis

In [ ]:
sentiment_pipeline = pipeline('sentiment-analysis', model='sangrimlee/bert-base-multilingual-cased-nsmc')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/932 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/712M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sangrimlee/bert-base-multilingual-cased-nsmc
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
sentiment_pipeline([
    "올드보이 ost는 진짜 ㅋㅋㅋ 영화속에서 유지태가 허밍으로 부르는 씬에서 진짜 고독함이라는 감정이 너무 억쎄게 밀려옴 ㅋㅋ",
    "편집이 왜이렇게 별로임 ㅋㅋ",
    "진짜 이 영상 너무 좋아요 감동받았어요",
    "배경음악 제목이 뭔가요?",
    "구독 했습니다 앞으로도 좋은 영상 부탁드려요"
])

[{'label': 'positive', 'score': 0.9809600114822388},
 {'label': 'negative', 'score': 0.992946207523346},
 {'label': 'positive', 'score': 0.9962183833122253},
 {'label': 'negative', 'score': 0.7881470322608948},
 {'label': 'positive', 'score': 0.8421984910964966}]

### zero-shot classification

In [ ]:
# zero_shot_pipeline = pipeline('zero-shot-classification', model = 'joeddav/xlm-roberta-large-xnli')

zero_shot_pipeline = pipeline('zero-shot-classification', model = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7')

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
sentence = '올드보이 ost는 진짜 ㅋㅋㅋ 영화속에서 유지태가 허밍으로 부르는 씬에서 진짜 고독함이라는 감정이 너무 억쎄게 밀려옴 ㅋㅋ'
# candidate_labels = ['칭찬', '비판', '정보', '질문', '광고', '감동' ]
candidate_labels = ['praise', 'criticism', 'information', 'question', 'advertisement', 'emotion']
hypothesis_template = '이 텍스트는 {}에 관한 내용입니다.'

zero_shot_pipeline(
    sentence,
    candidate_labels=candidate_labels,
    hypothesis_template=hypothesis_template
)

{'sequence': '올드보이 ost는 진짜 ㅋㅋㅋ 영화속에서 유지태가 허밍으로 부르는 씬에서 진짜 고독함이라는 감정이 너무 억쎄게 밀려옴 ㅋㅋ',
 'labels': ['emotion',
  'question',
  'information',
  'criticism',
  'praise',
  'advertisement'],
 'scores': [0.9341853857040405,
  0.03751838952302933,
  0.01512922067195177,
  0.007620461285114288,
  0.005001893267035484,
  0.00054466153960675]}

In [ ]:
comments = [
    "올드보이 ost 고독함이 너무 밀려옴 ㅋㅋ",
    "편집이 왜이렇게 별로임", 
    "배경음악 제목이 뭔가요?",
    "진짜 이 영상 너무 좋아요 감동받았어요",
    "구독 했습니다 앞으로도 좋은 영상 부탁드려요",
    "이 채널 진짜 최고입니다 매번 잘 보고 있어요",
    "참고로 이 방법은 틀렸습니다 실제로는 ~입니다",
    "제 채널도 방문해주세요 구독 부탁드립니다",
]

In [ ]:
for comment in comments:
    topic = zero_shot_pipeline(comment, candidate_labels)
    top_label = topic['labels'][0]

    if top_label in ['question', 'advertisement']:
        sent_label = 'neutral'
    else:
        sent = sentiment_pipeline(comment)[0]
        sent_label = sent["label"]

    print(f'{comment} | 감성: {sent_label} | 토픽: {top_label}')

올드보이 ost 고독함이 너무 밀려옴 ㅋㅋ | 감성: positive | 토픽: emotion
편집이 왜이렇게 별로임 | 감성: neutral | 토픽: question
배경음악 제목이 뭔가요? | 감성: neutral | 토픽: question
진짜 이 영상 너무 좋아요 감동받았어요 | 감성: positive | 토픽: praise
구독 했습니다 앞으로도 좋은 영상 부탁드려요 | 감성: positive | 토픽: praise
이 채널 진짜 최고입니다 매번 잘 보고 있어요 | 감성: positive | 토픽: praise
참고로 이 방법은 틀렸습니다 실제로는 ~입니다 | 감성: negative | 토픽: information
제 채널도 방문해주세요 구독 부탁드립니다 | 감성: negative | 토픽: information
